# 12 · Segmentación de los municipios rurales

> **En BigQuery.** Lee `MUNICIPIO_ANIO` de `DATASET_DEFORESTACION`. Para el mapa, descarga del DANE los límites municipales (no los guarda). Deja en `DATASET_RESULTADOS` una tabla de paso, `SEGMENTACION_MUNICIPIOS`, que lee y borra el notebook 17. Lo demás se ve aquí. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** unos cinco minutos.
>
> Las tablas, gráficas y frases calculadas usan los años de la celda 1. Los nombres y las descripciones de los ocho segmentos, y los demás textos escritos a mano, describen la corrida del trabajo de grado (perfil 2021-2024, deforestación 2021-2025): si cambian los datos o los años, hay que revisarlos.

Este notebook agrupa los municipios rurales y rurales dispersos según su **perfil productivo y financiero**:

- cuánto bosque les queda;
- cuánto ganado tienen y de qué tamaño son los hatos;
- cuánto cosechan y qué cultivos;
- cuánto crédito de producción reciben y qué parte va a la ganadería.

Después mira cuánto deforesta cada grupo.

**La deforestación no entra en la segmentación.** Se deja por fuera a propósito. Si entrara, los grupos se armarían
en parte con la misma variable que después se quiere comparar, y la comparación sería circular. Así, la pregunta
queda limpia: *¿los municipios con un mismo perfil productivo deforestan parecido?*

## Pasos

1. **Datos**: qué municipios, qué variables y qué hacer con los valores que faltan.
2. **Exploración y preprocesamiento**: transformación y estandarización, dentro de un `Pipeline` de scikit-learn.
3. **¿Hay grupos? ¿Cuántos?**: el codo, la silueta (comparada con datos sin estructura) y la estabilidad por
   bootstrap.
4. **Robustez**: la estabilidad de cada segmento por bootstrap.
5. **Perfil de cada segmento**: centros, medianas y componentes principales.
6. **Validación externa**: la deforestación de cada segmento.
7. **Mapa** de los segmentos.
8. **Conclusiones y limitaciones.**

Todo es descriptivo. Los segmentos muestran qué perfiles hay y cuánto deforesta cada uno, no qué causa la
deforestación.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Dataset donde quedan los resultados de la segmentación y de los
# modelos. Es el mismo de arriba. Para tenerlos aparte, pon otro
# nombre: el primer notebook que escriba ahí crea el dataset.
DATASET_RESULTADOS = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Años -------------------------------------------------------------
# El perfil de cada municipio es el promedio de ANIO_INICIO a ANIO_FIN - 1,
# los años que tienen las tres fuentes (la EVA llega hasta 2024). La
# deforestación con que se comparan los segmentos va de ANIO_INICIO a
# ANIO_FIN. En el trabajo de grado, 2021 y 2025. ANIO_FIN tiene que ser el
# mismo ANIO_PRUEBA de los notebooks 13 a 17.
ANIO_INICIO = 2021
ANIO_FIN = 2025

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet scikit-learn==1.6.1 geopandas==1.1.1 seaborn pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "12_modelos_segmentacion"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    f"""Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= {ANIO_INICIO}".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

In [ ]:
import logging
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

logging.getLogger("matplotlib").setLevel(logging.WARNING)
sns.set_theme(style="whitegrid", font_scale=1.15, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

SEMILLA = 0                      # todo lo aleatorio (K-means, bootstrap, permutaciones) parte de esta semilla
RURALES = ["Rural", "Rural disperso"]
# Colores de los segmentos: orden fijo, validado para daltonismo en barras y cajas (pares vecinos)
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
anual = leer_tabla("municipio_anio")

# En BigQuery solo quedan las tablas de datos. Los resultados se ven en el
# notebook; los únicos que se guardan son los que lee un notebook siguiente
# (tablas de paso). Las borra el último notebook que las usa (el 16 o el 17)
# y, si la cadena no se termina, vencen solas a los DIAS_TABLAS_DE_PASO días.
DIAS_TABLAS_DE_PASO = 7


def guardar_resultado(df, nombre, indice=False, descripcion=None):
    """Guarda en DATASET_RESULTADOS una tabla de paso para un notebook siguiente."""
    from datetime import timedelta
    if isinstance(df, pd.Series):
        df = df.to_frame()
    guardar_tabla(df, nombre, (descripcion or f"Resultado del notebook {NOMBRE_NOTEBOOK}: {nombre.replace('_', ' ')}.")
                  + " Tabla de paso: la borra el último notebook que la usa; si no, vence sola a los "
                  + f"{DIAS_TABLAS_DE_PASO} días.", dataset=DATASET_RESULTADOS, indice=indice)
    t = client.get_table(nombre_completo(nombre, DATASET_RESULTADOS))
    t.expires = datetime.now(timezone.utc) + timedelta(days=DIAS_TABLAS_DE_PASO)
    client.update_table(t, ["expires"])
    return df

## 1. Datos: municipios y variables

**La pregunta.** ¿Qué municipios se segmentan y con qué variables?

**Cómo se hace.**

- **Municipios.** Los rurales y rurales dispersos de la clasificación del DNP, los mismos del análisis del panel
  municipal. Quedan por fuera los dos que no tienen celdas en la grilla de 5 km: sin área no se pueden calcular las
  intensidades.
- **Periodo.** El promedio de 2021 a 2024, los años que tienen las tres fuentes (la EVA llega hasta 2024). Así se
  evita también 2025 en el ICA, el año en que el inventario de bovinos se reasigna entre municipios. Promediar cuatro
  años quita el ruido de un año atípico.
- **Variables.** Son diez, en cuatro bloques:
  - territorio: el bosque;
  - ganadería: la densidad y el tamaño del hato;
  - agricultura: cuánto se cosecha y qué cultivos;
  - crédito: cuánto llega y qué parte va a la ganadería.

  Todas son **relativas** (por km², % del municipio o % de un total), para que el tamaño del municipio no mande. Dos
  municipios no deben quedar juntos solo por ser grandes.
- **Qué se deja por fuera y por qué.**
  - La **deforestación**: es lo que se compara después, en la sección 6.
  - El **área del municipio**: ya está descontada en las intensidades. Solo se usa para describir los segmentos.
  - Las **fincas por km²**: salen de dividir los bovinos por km² entre los bovinos por finca, así que repetirían
    información.
  - El **crédito a palma, cacao, café y arroz**: sigue de cerca a la participación de cada cultivo. Meterlo
    contaría dos veces lo mismo.
- **Valores que faltan.**
  - Si un municipio no cosechó nada, la participación de cada cultivo es 0.
  - Si no recibió crédito, la del crédito ganadero es 0.
  - Si no tiene bovinos, el hato es 0.
  - Lo poco que queda vacío (bovinos sin fincas registradas) se llena con la mediana, dentro del preprocesamiento.

In [ ]:
rurales = anual[anual.ruralidad.isin(RURALES)]
# Columna del panel que se promedia para cada insumo de las variables
INSUMOS = {"bovinos": "bovinos", "fincas": "fincas_bovinas", "cosechada": "area_cosechada_ha",
           "palma": "cosechada_palma_ha", "cacao": "cosechada_cacao_ha", "cafe": "cosechada_cafe_ha",
           "arroz": "cosechada_arroz_ha", "credito": "credito_produccion_mm",
           "credito_gan": "credito_ganaderia_produccion_mm"}


def variables(t):
    """Las diez variables de la segmentacion. t trae, por municipio, el area, el bosque y los INSUMOS."""
    km2 = t.area_ha / 100
    cosechada = t.cosechada.where(t.cosechada > 0)
    credito = t.credito.where(t.credito > 0)
    # Sin bovinos el hato es 0; con bovinos pero sin fincas queda vacio y se imputa
    hato = (t.bovinos / t.fincas.where(t.fincas > 0)).mask(t.bovinos == 0, 0)
    return pd.DataFrame({
        "bosque": 100 * t.bosque_ha / t.area_ha,
        "bovinos": t.bovinos / km2,
        "hato": hato,
        "agricola": 100 * t.cosechada / t.area_ha,
        # Sin area cosechada (o sin credito) la participacion de cada cultivo (o de la ganaderia) es 0
        "palma": (100 * t.palma / cosechada).fillna(0),
        "cacao": (100 * t.cacao / cosechada).fillna(0),
        "cafe": (100 * t.cafe / cosechada).fillna(0),
        "arroz": (100 * t.arroz / cosechada).fillna(0),
        "credito": t.credito / km2,
        "credito_gan": (100 * t.credito_gan / credito).fillna(0),
    })


fijos = (rurales.groupby("cod_dane")
         .agg(area_ha=("area_grilla_ha", "first"), bosque_ha=("bosque_ideam_inicial_ha", "first"),
              nombre=("nombre_municipio", "first"), departamento=("nombre_departamento", "first"),
              region=("region", "first"), ruralidad=("ruralidad", "first"))
         .astype({"nombre": str, "departamento": str, "region": str, "ruralidad": str}))
promedio = rurales[rurales.anio.between(ANIO_INICIO, ANIO_FIN - 1)].groupby("cod_dane")[list(INSUMOS.values())].mean()
promedio.columns = list(INSUMOS)
base = fijos.join(promedio)

sin_area = base.area_ha.isna()
print(f"Municipios rurales y rurales dispersos: {len(base)}")
print(f"  sin celdas de la grilla (sin área ni bosque), quedan por fuera: {base[sin_area].nombre.tolist()}")
base = base[~sin_area]
X = variables(base)
VARIABLES = list(X.columns)
print(f"  en la segmentación: {len(X)}  {base.ruralidad.value_counts().to_dict()}")
print(f"  sin área cosechada en {ANIO_INICIO}-{ANIO_FIN - 1}: {(base.cosechada == 0).sum()}; sin crédito de producción: "
      f"{(base.credito == 0).sum()}; sin bovinos: {(base.bovinos == 0).sum()}")
print(f"  valores vacíos que se imputan con la mediana: {X.isna().sum()[X.isna().sum() > 0].to_dict()}")

# Deforestacion: NO entra en la segmentacion; se usa despues para comparar los segmentos
periodo_def = rurales[rurales.anio.between(ANIO_INICIO, ANIO_FIN)].groupby("cod_dane").agg(
    tasa=("tasa_def_ideam", "mean"), def_ha=("def_ideam_ha", "mean"),
    def_total_ha=("def_ideam_ha", "sum"), gfw_ha=("alertas_gfw_ha", "mean")).reindex(X.index)
km2 = base.area_ha / 100
deforestacion = pd.DataFrame({
    "tasa": 100 * periodo_def.tasa,                          # % del bosque perdido por año
    "def_100km2": 100 * periodo_def.def_ha / km2,            # ha deforestadas por 100 km2 al año
    "gfw_100km2": 100 * periodo_def.gfw_ha / km2,            # ha con alertas de GFW por 100 km2 al año
    "def_total_ha": periodo_def.def_total_ha,                # ha deforestadas 2021-2025
})

NOMBRES = {
    "bosque": f"Bosque (% del municipio, {ANIO_INICIO})", "bovinos": "Bovinos por km²", "hato": "Bovinos por finca",
    "agricola": "Área cosechada (% del municipio)", "palma": "Palma (% del área cosechada)",
    "cacao": "Cacao (% del área cosechada)", "cafe": "Café (% del área cosechada)",
    "arroz": "Arroz (% del área cosechada)", "credito": "Crédito de producción (M$ por km² al año)",
    "credito_gan": "Crédito ganadero (% del crédito de producción)",
}
CORTOS = {"bosque": "Bosque", "bovinos": "Bovinos/km²", "hato": "Bovinos/finca", "agricola": "Área cosechada",
          "palma": "Palma", "cacao": "Cacao", "cafe": "Café", "arroz": "Arroz", "credito": "Crédito/km²",
          "credito_gan": "Crédito ganadero"}
pd.DataFrame({
    "variable": [NOMBRES[c] for c in VARIABLES],
    "qué describe": ["Cuánto del territorio sigue en bosque", "Densidad del ganado", "Tamaño medio del hato",
                     "Peso de la agricultura en el territorio", "Qué se cultiva", "Qué se cultiva",
                     "Qué se cultiva", "Qué se cultiva", "Cuánto crédito llega", "Para qué va el crédito"],
    "fuente": ["IDEAM"] + ["ICA"] * 2 + ["EVA"] * 5 + ["FINAGRO"] * 2,
    "años": [f"{ANIO_INICIO}"] + [f"promedio {ANIO_INICIO}-{ANIO_FIN - 1}"] * 9,
    "transformación": ["ninguna", "log(1 + x)", "log(1 + x)", "log(1 + x)"] + ["ninguna"] * 4
                      + ["log(1 + x)", "ninguna"],
}).set_index("variable")

## 2. Exploración y preprocesamiento

**La pregunta.** ¿Cómo se distribuyen las variables y qué hay que hacerles antes de segmentar?

**Cómo se hace.** Se mira la distribución de cada variable y la correlación entre ellas. K-means agrupa por
distancias, así que importan dos cosas:

- que ninguna variable domine por su escala;
- que unos pocos valores extremos no arrastren a los grupos.

**Cómo leer los gráficos.** Los histogramas cuentan municipios por rango de valor. La matriz trae la correlación de
Spearman entre cada par de variables: si es cercana a 1 o a −1, las dos dicen casi lo mismo.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, c in zip(axes.flat, VARIABLES):
    ax.hist(X[c].dropna(), bins=40, color=PALETA[0], edgecolor="white", linewidth=0.5)
    ax.set_title(CORTOS[c])
    ax.set_xlabel(NOMBRES[c], fontsize=10)
axes[0, 0].set_ylabel("municipios")
axes[1, 0].set_ylabel("municipios")
fig.suptitle("Distribución de cada variable en los municipios rurales (sin transformar)", weight="bold", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .96))
plt.show()

pd.DataFrame({"mediana": X.median(), "media": X.mean(), "asimetría": X.skew(),
              "% de municipios en 0": 100 * (X == 0).mean()}).rename(index=NOMBRES).round(2)

In [ ]:
correlacion = X.rename(columns=CORTOS).corr("spearman")
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlacion, mask=np.triu(np.ones_like(correlacion, dtype=bool), k=1), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"label": "correlación de Spearman", "shrink": .8})
ax.set_title("Correlación entre las variables de la segmentación")
plt.show()

**Qué se ve.**

- Casi todas las variables tienen una **cola larga a la derecha**: muchos municipios con poco y unos pocos con
  mucho. La más asimétrica es el área cosechada (asimetría de 4,2). Le siguen el arroz, la palma, el crédito por km²
  y el cacao, entre 2,9 y 3,6.
- Los cultivos tienen **muchos ceros**:
  - el 86 % de los municipios no cosecha palma;
  - el 69 % no cosecha arroz;
  - cerca del 41 % no cosecha cacao, y otro tanto no cosecha café.

  Son cultivos que están en unos municipios y no en otros, y eso es justo lo que puede formar grupos.
- Ningún par de variables está tan correlacionado como para repetir información. Las correlaciones más fuertes son:
  - la del área cosechada con el crédito por km² (0,70): donde se cultiva más llega más crédito;
  - la del bosque con los bovinos por km² (−0,67): donde hay más ganado queda menos bosque.

### Preprocesamiento

**Cómo se hace.** Todo va en un `Pipeline` de scikit-learn, para que la transformación quede en un solo objeto que se
puede volver a aplicar igual. Tiene tres pasos:

1. **Imputación** de los pocos valores vacíos con la mediana.
2. **log(1 + x)** a las cuatro intensidades con cola larga: bovinos por km², bovinos por finca, área cosechada y
   crédito por km².
   - El logaritmo hace que pasar de 10 a 100 pese lo mismo que pasar de 100 a 1.000.
   - El +1 deja los ceros en 0.
   - Las participaciones (de 0 a 100 %) y el bosque se dejan como están.
3. **Estandarización**: cada variable queda con media 0 y desviación estándar 1, para que todas pesen lo mismo en la
   distancia.

**Cómo leer el gráfico.** Cada caja es una variable ya transformada. Los puntos sueltos son municipios lejos del
resto.

In [ ]:
LOG = ["bovinos", "hato", "agricola", "credito"]
preproceso = make_pipeline(
    SimpleImputer(strategy="median"),
    ColumnTransformer([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), LOG)],
                      remainder="passthrough", verbose_feature_names_out=False),
    StandardScaler(),
).set_output(transform="pandas")
Z = preproceso.fit_transform(X)[VARIABLES]
Zv = Z.to_numpy()

fig, ax = plt.subplots(figsize=(14, 8))
sns.boxplot(data=Z.rename(columns=CORTOS), orient="h", color=PALETA[0], fliersize=3, ax=ax)
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Las variables después de transformar y estandarizar", xlabel="desviaciones estándar respecto a la media")
plt.show()
print("Municipios a más de 3 desviaciones estándar de la media, por variable:")
print((Z.abs() > 3).sum().rename(CORTOS).to_string())

**Qué se ve.**

- Con el logaritmo, ningún municipio queda a más de 3 desviaciones estándar en bosque, bovinos por km² o crédito.
- Los que siguen lejos del resto son de dos tipos:
  - municipios donde un cultivo pesa mucho: 29 en palma, 20 en arroz y 16 en cacao;
  - municipios con hatos muy grandes: 15.

No son errores: son municipios donde un cultivo o la ganadería dominan, y eso es justo lo que la segmentación debe
ver. Por eso se dejan.

## 3. ¿Hay grupos? ¿Cuántos?

**La pregunta.** ¿Los municipios forman grupos de verdad, o cualquier partición sería arbitraria? Y si hay grupos,
¿cuántos?

**Cómo se hace.** Se corre K-means con k = 2 a 10 segmentos y se miran tres criterios. Cada k se corre con 50
arranques distintos, para no quedarse en una mala solución. Ningún criterio basta solo, así que se miran juntos:

- **Inercia (codo).** Es la suma de las distancias de cada municipio al centro de su grupo. Siempre baja al subir k.
  Se busca el punto donde deja de bajar rápido.
- **Silueta.** Va de −1 a 1 y mide qué tan cerca está cada municipio de su grupo frente al grupo vecino.
  - Se compara con la silueta de **datos sin estructura**: las mismas variables, pero cada una barajada por
    separado. Esos datos tienen las mismas distribuciones, pero ningún grupo real.
  - Si la silueta de los datos no supera a la de los barajados, los "grupos" serían un artificio.
- **Estabilidad por bootstrap.**
  - Se remuestrean los municipios 100 veces, se vuelve a segmentar y se compara con la segmentación original.
  - La comparación usa el índice de Rand ajustado (ARI): 1 es idéntica y 0 es como al azar.
  - Una segmentación útil no debe cambiar mucho con una muestra un poco distinta. Se toma 0,85 como referencia de
    "estable".

**Cómo leer el gráfico.** Cada panel es un criterio, con k en el eje horizontal.

- En la silueta, la franja gris es lo que da el azar.
- En la estabilidad, la franja azul va del percentil 10 a la media de los 100 remuestreos.

In [ ]:
def estabilidad(Z, k, B=100):
    """ARI entre la segmentacion de todos los datos y la de B remuestreos bootstrap (una cifra por remuestreo)."""
    rng = np.random.default_rng(SEMILLA)
    completa = KMeans(k, n_init=20, random_state=SEMILLA).fit_predict(Z)
    aris = []
    for b in range(B):
        muestra = rng.choice(len(Z), len(Z), replace=True)
        modelo = KMeans(k, n_init=10, random_state=b).fit(Z[muestra])
        aris.append(adjusted_rand_score(completa, modelo.predict(Z)))
    return np.array(aris)


def silueta_sin_estructura(Z, k, P=20):
    """Silueta de K-means cuando cada variable se baraja por separado: mismas distribuciones, sin grupos."""
    rng = np.random.default_rng(SEMILLA)
    siluetas = []
    for p in range(P):
        barajado = np.column_stack([rng.permutation(columna) for columna in Z.T])
        etiquetas = KMeans(k, n_init=10, random_state=p).fit_predict(barajado)
        siluetas.append(silhouette_score(barajado, etiquetas))
    return np.array(siluetas)


filas = []
for k in range(2, 11):
    modelo = KMeans(k, n_init=50, random_state=SEMILLA).fit(Zv)
    nula = silueta_sin_estructura(Zv, k)
    ari = estabilidad(Zv, k)
    filas.append({"k": k, "inercia": modelo.inertia_, "silueta": silhouette_score(Zv, modelo.labels_),
                  "silueta sin estructura": nula.mean(), "silueta sin estructura (máx.)": nula.max(),
                  "estabilidad (ARI medio)": ari.mean(), "estabilidad (ARI, percentil 10)": np.percentile(ari, 10),
                  "segmento más pequeño": np.bincount(modelo.labels_).min()})
indices = pd.DataFrame(filas).set_index("k")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
ks = indices.index
graficos = [("inercia", "Inercia (codo)", "suma de distancias al centro, menor es mejor"),
            ("silueta", "Silueta", "mayor es mejor"),
            ("estabilidad (ARI medio)", "Estabilidad por bootstrap", "ARI, mayor es mejor")]
for ax, (c, titulo, eje) in zip(axes, graficos):
    ax.plot(ks, indices[c], marker="o", color=PALETA[0], lw=2, ms=8, label="datos")
    ax.set(title=titulo, xlabel="número de segmentos (k)", ylabel=eje, xticks=ks)
axes[1].plot(ks, indices["silueta sin estructura"], marker="o", color="grey", lw=2, ms=8, ls="--",
             label="datos sin estructura (media)")
axes[1].fill_between(ks, indices["silueta sin estructura"], indices["silueta sin estructura (máx.)"],
                     color="grey", alpha=.2, label="datos sin estructura (hasta el máximo)")
axes[1].legend(fontsize=11)
axes[2].fill_between(ks, indices["estabilidad (ARI, percentil 10)"], indices["estabilidad (ARI medio)"],
                     color=PALETA[0], alpha=.15, label="del percentil 10 a la media")
axes[2].axhline(0.85, color="grey", ls=":", lw=1.5, label="0,85")
axes[2].legend(fontsize=11)
fig.suptitle("¿Cuántos segmentos? Codo, silueta y estabilidad de K-means para k = 2 a 10", weight="bold",
             fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .93))
plt.show()
indices.round(3)

### La decisión: 8 segmentos

**Qué se ve.**

- **Sí hay grupos, pero no muy marcados.**
  - Desde k = 5, la silueta de los datos está entre 0,24 y 0,27. Es el doble de la de los datos barajados
    (alrededor de 0,13) y supera su máximo. Los grupos no son un artificio.
  - Aun así, 0,26 está en el límite inferior de lo que la referencia usual (Kaufman y Rousseeuw, 1990) llama una
    estructura débil. Los segmentos existen, pero sus fronteras no son nítidas: hay municipios a medio camino entre
    dos segmentos.
- **k = 2 no sirve, aunque tenga la silueta más alta.** Su silueta (0,306) es igual al máximo de los datos barajados
  (0,307). Además, solo separa los municipios de selva del resto.
- **El codo no es un quiebre limpio.** Hasta k = 7, cada segmento nuevo baja la inercia en más de 340. Desde k = 8
  baja cada vez menos: 205, 127 y 82.
- **La estabilidad decide.**
  - Aparte de k = 2, solo k = 7 (0,87) y k = 8 (0,88) pasan de 0,85.
  - k = 8 es el más estable de los dos y el que tiene el percentil 10 más alto (0,84): aun en los peores
    remuestreos, la segmentación casi no cambia.
  - Con k = 6, en cambio, la estabilidad cae a 0,64.

Se eligen **8 segmentos**. Frente a 7, el octavo separa del resto de los ganaderos a los municipios con mucho bosque y
poco ganado.

**Lectura: contexto técnico, reglas y supuestos de la segmentación (REQ-04, CDM-6).**

*Las cifras de las lecturas de este notebook están copiadas de las tablas de la corrida del 20/11/2026 y pueden diferir en decimales si se vuelve a ejecutar con otras versiones de las librerías.*

- **Qué es y qué no es.** Es un diagnóstico **no supervisado y descriptivo**: agrupa municipios por su perfil productivo y financiero y después mira cuánto deforesta cada grupo. **No predice**: responde otra pregunta
  (¿qué tipos de municipio existen y cuál concentra la deforestación?). Por eso no tiene una búsqueda de hiperparámetros con validación cruzada: lo único que se decide es **k**, y se decide con tres criterios medidos (abajo).
- **Configuración fija.** K-means con distancia euclidiana, k = 8, 100 inicializaciones y semilla 0 (50 inicializaciones en el barrido de k); diez variables relativas (por km², % del municipio o % de un total) promediadas en 2021-2024; mediana para los pocos vacíos;
  `log1p` en las cuatro intensidades de cola larga (bovinos por km², bovinos por finca, área cosechada, crédito por km²) y estandarización, todo dentro de un `Pipeline` para que sea reproducible (CDM-6).
- **Cómo se eligió k (el equivalente de "la búsqueda").** Se barrió k de 2 a 10 con tres reglas: (1) la silueta tiene que superar a la de datos **barajados** sin estructura (0,262 frente a 0,132 de media y 0,143 de máximo, con k = 8); (2) la estabilidad por bootstrap
  (ARI medio 0,879 y percentil 10 de 0,838, el más alto entre los k desde 3); (3) que ningún segmento quede diminuto (el más pequeño tiene 35 municipios). k = 2 tiene la silueta más alta (0,306) pero es igual al máximo de los datos barajados, así que no se acepta.
- **Supuestos que hay que declarar.**
  1. K-means supone grupos más o menos **compactos y de dispersión parecida**; un grupo alargado o muy disperso se parte o se mezcla (es lo que pasa con el bosque con ganadería dispersa, ver la lectura de solidez).
  2. **Las diez variables pesan lo mismo**, y cuatro son cultivos; con otra selección los segmentos pueden cambiar.
  3. El perfil es **estático** (promedio 2021-2024): un municipio no cambia de segmento de un año a otro.
  4. La deforestación **se deja fuera a propósito** para que la comparación posterior sea una validación externa y no circular.
  5. Los ceros (sin cultivos, sin crédito, sin bovinos) se tratan como valores reales, aunque en la Amazonía pueden ser falta de reporte.
- **Cómo se conecta con el documento.** Cada segmento se describe con variables que cualquier persona entiende (bosque, ganado, cultivo dominante, crédito), así que cumple REQ-04 (interpretabilidad, sin cajas negras) y es el soporte de ODM-2 (cruce de financiamiento y producción) a nivel de tipo de municipio.

In [ ]:
K = 8
km = KMeans(K, n_init=100, random_state=SEMILLA).fit(Zv)
# Las etiquetas de K-means son arbitrarias: se numeran de 1 a K de mas a menos bosque (mediana)
orden = X.bosque.groupby(km.labels_).median().sort_values(ascending=False).index
RECODIFICAR = {vieja: nueva for nueva, vieja in enumerate(orden, start=1)}
segmento = pd.Series(km.labels_, index=X.index).map(RECODIFICAR).rename("segmento")
SEGS = list(range(1, K + 1))
COLOR = dict(zip(SEGS, PALETA))

perfil = X.groupby(segmento).median()
perfil.insert(0, "municipios", segmento.value_counts().sort_index())
perfil.rename(columns=CORTOS).round(1)

### Los nombres de los segmentos

Los números que pone K-means no significan nada, así que los segmentos se numeran de más a menos bosque. Los
nombres se pusieron después de mirar los perfiles de la sección 5, y cada uno describe el rasgo que más distingue
al grupo.

In [ ]:
SEGMENTOS = {
    1: "1. Selva sin actividad agropecuaria",
    2: "2. Bosque con ganadería dispersa",
    3: "3. Cacao y bosque",
    4: "4. Cafetero",
    5: "5. Minifundio con mucho crédito",
    6: "6. Palmero",
    7: "7. Ganadero",
    8: "8. Arrocero",
}
nombre_segmento = segmento.map(SEGMENTOS)
ETIQUETAS = [SEGMENTOS[s] for s in SEGS]
COLOR_NOMBRE = {SEGMENTOS[s]: COLOR[s] for s in SEGS}

## 4. ¿Qué tan robusta es la segmentación?

**La pregunta.** ¿Cada segmento es real, o aparece solo con estos municipios exactos y se deshace con una muestra un
poco distinta?

**Cómo se hace.** Se mide la **estabilidad de cada segmento** (Hennig, 2007):

1. Se remuestrean los municipios 200 veces y en cada remuestreo se vuelve a segmentar.
2. Para cada segmento original se busca el más parecido en la nueva segmentación.
3. El parecido se mide con el índice de Jaccard: los municipios en común sobre los municipios en cualquiera de los
   dos. Va de 0 (ninguno en común) a 1 (los mismos municipios).
4. Se promedia el Jaccard de los 200 remuestreos.

La lectura usual es:

- más de 0,85: muy estable;
- de 0,75 a 0,85: estable;
- de 0,6 a 0,75: un patrón con dudas;
- menos de 0,5: el segmento se disuelve.

La estabilidad de la sección 3 mide la segmentación completa. Esta mide cada segmento por separado.

In [ ]:
def jaccard_bootstrap(Z, etiquetas, k, B=200):
    """Estabilidad de cada segmento (Hennig, 2007): Jaccard medio con el segmento mas parecido en B bootstrap."""
    rng = np.random.default_rng(SEMILLA)
    jaccard = np.full((B, k), np.nan)
    for b in range(B):
        muestra = rng.choice(len(Z), len(Z), replace=True)
        modelo = KMeans(k, n_init=10, random_state=b).fit(Z[muestra])
        unicos = np.unique(muestra)
        nuevas, viejas = modelo.predict(Z[unicos]), etiquetas[unicos]
        for s in range(k):
            en_s = viejas == s
            if en_s.any():
                jaccard[b, s] = max((en_s & (nuevas == t)).sum() / (en_s | (nuevas == t)).sum() for t in range(k))
    return np.nanmean(jaccard, axis=0)



etiquetas0 = segmento.to_numpy() - 1
estab_seg = pd.DataFrame({"municipios": segmento.value_counts().sort_index().values,
                          "Jaccard medio (bootstrap)": jaccard_bootstrap(Zv, etiquetas0, K)}, index=ETIQUETAS)
estab_seg["lectura"] = pd.cut(estab_seg["Jaccard medio (bootstrap)"], [0, .5, .6, .75, .85, 1],
                              labels=["se disuelve", "débil", "patrón con dudas", "estable", "muy estable"])
estab_seg.round(2)

**Qué se ve.**

- **Los ocho segmentos son estables.** Ninguno se disuelve ni queda como un patrón con dudas.
- **Seis son muy estables**, con un Jaccard por encima de 0,85:
  - los más nítidos son la selva (0,97) y el palmero (0,96);
  - les siguen el cafetero (0,93), el arrocero (0,92), el cacaotero (0,91) y el minifundio (0,90).
- **Los otros dos son estables, pero menos nítidos**: el ganadero (0,85) y el de bosque con ganadería dispersa
  (0,81). Con una muestra un poco distinta, algunos de sus municipios cambian de segmento.

En resumen, los segmentos no dependen de qué municipios exactos entren en la muestra. El de bosque con ganadería
dispersa y el ganadero deben leerse como tipos de municipio con fronteras menos nítidas.

### Solidez de cada perfil: ¿qué tan separado está de su vecino más parecido?

**La pregunta.** La estabilidad de arriba dice si un segmento reaparece con otra muestra. No dice si sus municipios están **bien dentro** del segmento o en la
frontera con otro. Para eso se usa la **silueta** de cada municipio (Rousseeuw, 1987): compara qué tan cerca está de su propio segmento con qué tan cerca está del
segmento vecino. Va de -1 a 1; un valor negativo significa que el municipio está más cerca del vecino que de su propio segmento.

**Cómo se hace.**

1. Se calcula la silueta de cada municipio con las mismas variables transformadas de la segmentación.
2. Por segmento se resume: silueta media y % de municipios con silueta negativa.
3. Se identifica el **segmento vecino** (el de centro más cercano) y la distancia entre los dos centros.

Lectura usual de la silueta media (Kaufman y Rousseeuw, 1990): más de 0,50, estructura fuerte; de 0,26 a 0,50, razonable; 0,25 o menos, débil. Es una guía, no una prueba.

In [ ]:
from sklearn.metrics import silhouette_samples
from scipy.spatial.distance import cdist

sil = silhouette_samples(Zv, etiquetas0)
centros = np.vstack([Zv[etiquetas0 == s].mean(axis=0) for s in range(K)])
dist_centros = cdist(centros, centros)
filas = []
for s in range(K):
    m = etiquetas0 == s
    d = cdist(Zv[m], centros)
    d[:, s] = np.inf
    vecino_s = int(np.bincount(d.argmin(axis=1), minlength=K).argmax())  # segmento vecino mas frecuente entre sus municipios
    filas.append({"segmento": ETIQUETAS[s], "municipios": int(m.sum()), "silueta media": sil[m].mean(),
                  "% con silueta negativa": 100 * (sil[m] < 0).mean(),
                  "segmento vecino": ETIQUETAS[vecino_s], "distancia entre centros": dist_centros[s, vecino_s],
                  "Jaccard medio (bootstrap)": estab_seg.iloc[s, 1]})
solidez = pd.DataFrame(filas).set_index("segmento")
solidez["lectura de la silueta"] = pd.cut(solidez["silueta media"], [-1, 0.25, 0.50, 1], labels=["débil", "razonable", "fuerte"])
display(solidez.round(2))
print(f"Silueta global: {sil.mean():.3f}. Municipios con silueta negativa: {int((sil < 0).sum())} de {len(sil)} ({100 * (sil < 0).mean():.0f} %).")
mas_solido, menos_solido = solidez["silueta media"].idxmax(), solidez["silueta media"].idxmin()
print(f"El perfil mejor separado es «{mas_solido}» (silueta {solidez.loc[mas_solido, 'silueta media']:.2f}); el menos separado, «{menos_solido}» "
      f"(silueta {solidez.loc[menos_solido, 'silueta media']:.2f}, {solidez.loc[menos_solido, '% con silueta negativa']:.0f} % de sus municipios más cerca de «{solidez.loc[menos_solido, 'segmento vecino']}»).")


**Qué se ve.** Léase junto con la estabilidad: un perfil puede ser estable (siempre reaparece) y aun así tener una frontera difusa con su vecino. Los perfiles con silueta media baja
y varios municipios con silueta negativa deben leerse como **tipos con frontera difusa**: sus estrategias se aplican a los municipios del centro del grupo, y los de la frontera pueden
requerir mirar también el perfil vecino. Las cifras exactas están en la tabla (se calculan al ejecutar la celda y no se copian a mano aquí).

**Lectura: qué tan firme es cada perfil y cómo usarlo (robustez por grupo).**

Se combinan tres señales: la estabilidad por bootstrap (¿el grupo reaparece si cambia la muestra?), la silueta (¿está separado de su vecino?) y el % de municipios con silueta negativa (¿cuántos están en realidad más cerca de otro perfil?).

- **Firmes: se puede aplicar la estrategia a todo el grupo.** Selva (Jaccard 0,97, silueta 0,45, 0 % negativos), palmero (0,96, 0,35, 5 %) y arrocero (0,92, 0,34, 3 %). Están lejos de su vecino más cercano (distancia entre centros de 3,4 a 4,1) y casi ningún municipio queda a medio camino.
- **Firmes, pero vecinos entre sí: cafetero y minifundio** (0,93 y 0,90; silueta 0,26 cada uno). Sus centros son los más cercanos de todos los pares (2,32), pero casi ninguno de sus municipios es ambiguo (1 de 123 y 0 de 142 con silueta negativa): son
  perfiles contiguos, no confundidos. Se pueden tratar por separado.
- **Intermedio: cacao y bosque** (0,91, silueta 0,25, 4 de 47 con silueta negativa). Es estable, pero un municipio de cada diez está más cerca del bosque con ganadería dispersa.
- **Frontera difusa: bosque con ganadería dispersa y ganadero** (Jaccard 0,81 y 0,85; silueta 0,12 y 0,21). Son vecinos entre sí (distancia 2,38). El de bosque con ganadería dispersa tiene **14 de los 26 municipios con silueta negativa de todo el país rural**
  (16 % del grupo), y es justamente el que aporta la mitad de las hectáreas deforestadas (53 %).
- **Qué significa para el uso.** Para los grupos firmes, el perfil basta como "tipo de municipio" en una ficha. Para los de frontera difusa, la etiqueta **no debe usarse sola**: hay que mirar también el perfil vecino en los municipios con silueta negativa. Es una limitación para declarar tal cual: el grupo donde más importa decidir es el de contornos menos nítidos.

## 5. Perfil de cada segmento

**La pregunta.** ¿Qué distingue a cada segmento?

**Cómo se hace.** Hay tres formas de verlo:

- El **centro** de cada segmento en las variables estandarizadas: cuántas desviaciones estándar está por encima o
  por debajo del municipio rural promedio.
- La **mediana** de cada variable, en sus unidades originales.
- Los municipios en los **dos primeros componentes principales**, que resumen las diez variables en dos ejes.

**Cómo leer el mapa de calor.** En rojo, el segmento está por encima del promedio en esa variable; en azul, por
debajo. El número es la distancia en desviaciones estándar.

**Cómo leer los componentes principales.** El primer panel muestra hacia dónde crece cada variable. En los demás,
cada segmento va en su color sobre el resto de municipios en gris.

In [ ]:
centros = Z.groupby(nombre_segmento).mean().rename(columns=CORTOS)
fig, ax = plt.subplots(figsize=(17, 7.5))
sns.heatmap(centros, annot=True, fmt=".1f", cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5, linewidths=0.5, ax=ax,
            cbar_kws={"label": "desviaciones estándar respecto al municipio rural promedio"})
ax.set(title="Perfil de cada segmento: el centro de cada grupo en las variables estandarizadas", ylabel="",
       xlabel="")
plt.show()

mediana = X.groupby(nombre_segmento).median().rename(columns=NOMBRES)
mediana.insert(0, "municipios", nombre_segmento.value_counts())
mediana.insert(1, "% de los municipios", 100 * nombre_segmento.value_counts(normalize=True))
mediana.insert(2, "área mediana (km²)", (base.area_ha / 100).groupby(nombre_segmento).median())
mediana.round(1).T

In [ ]:
pca = PCA(random_state=SEMILLA).fit(Zv)
coordenadas = pca.transform(Zv)[:, :2]
var = 100 * pca.explained_variance_ratio_
print("Varianza explicada por cada componente (%):", np.round(var, 1))
print(f"Los dos primeros explican el {var[:2].sum():.0f} %.")

fig, axes = plt.subplots(3, 3, figsize=(21, 19), sharex=True, sharey=True)
ax = axes.flat[0]
ax.scatter(*coordenadas.T, s=10, color="#c8c8c8")
# Las flechas se escalan para que la mas larga quepa dentro del panel
margen = min(-coordenadas.min(axis=0).max(), coordenadas.max(axis=0).min())
escala = 0.75 * margen / np.linalg.norm(pca.components_[:2], axis=0).max()
for c, (x, y) in zip(VARIABLES, pca.components_[:2].T):
    ax.annotate("", xy=(escala * x, escala * y), xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": "black", "lw": 1.5})
    ax.text(1.15 * escala * x, 1.15 * escala * y, CORTOS[c], fontsize=11, ha="center", va="center",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 1})
ax.set_title("Hacia dónde crece cada variable")
for ax, s in zip(axes.flat[1:], SEGS):
    ax.scatter(*coordenadas.T, s=10, color="#dcdcdc")
    ax.scatter(*coordenadas[segmento.values == s].T, s=22, color=COLOR[s], edgecolor="white", linewidth=0.4)
    ax.set_title(SEGMENTOS[s])
for ax in axes[-1]:
    ax.set_xlabel(f"componente 1 ({var[0]:.0f} %)")
for ax in axes[:, 0]:
    ax.set_ylabel(f"componente 2 ({var[1]:.0f} %)")
fig.suptitle("Los municipios en los dos primeros componentes principales, un segmento por panel", weight="bold",
             fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .97))
plt.show()
pd.DataFrame(pca.components_[:3].T, index=[NOMBRES[c] for c in VARIABLES],
             columns=["componente 1", "componente 2", "componente 3"]).round(2)

### Dónde están los segmentos

**Cómo se hace.** Para cada segmento se calcula qué % de sus municipios está en cada región, y cuáles son los
departamentos con más municipios. La región **no** entró en la segmentación. Si los segmentos se parecen a regiones,
es porque los perfiles productivos tienen geografía.

In [ ]:
region = pd.crosstab(nombre_segmento, base.region, normalize="index").mul(100).round(0)
region["departamentos con más municipios"] = [
    ", ".join(f"{d} ({n})" for d, n in base.departamento[nombre_segmento == e].value_counts().head(4).items())
    for e in region.index]
region

**Qué se ve.** Cada segmento tiene un rasgo que lo distingue. Las cifras son medianas.

1. **Selva sin actividad agropecuaria** (64 municipios).
   - Casi todo es bosque (89 %).
   - Casi no tiene ganado (0,1 bovinos por km²), cultivos (0,5 % del municipio) ni crédito (0,3 M$ por km²).
   - Son municipios grandes (1.900 km²) del Pacífico, como Chocó y Nariño, y de la Amazonía y la Orinoquía
     oriental, como Amazonas, Guainía y Vaupés. 18 de los 64 son corregimientos departamentales.
2. **Bosque con ganadería dispersa** (86).
   - Mucho bosque (55 %) y algo de ganado (15 bovinos por km², en hatos de 32).
   - Muy poca agricultura (1,8 %) y poco crédito (4,8 M$ por km²), del que el 40 % va a la ganadería.
   - Es el segmento más mezclado en geografía. Tiene el piedemonte amazónico (Caquetá, Guaviare, sur del Meta y
     Putumayo) y también municipios con bosque de los Andes y la costa.
3. **Cacao y bosque** (47).
   - El cacao es el 39 % del área cosechada, y el bosque, el 37 % del municipio.
   - Está sobre todo en Santander, Antioquia y Boyacá.
4. **Cafetero** (123).
   - El café es el 53 % del área cosechada.
   - Está en la cordillera: Santander, Cauca, Huila y Antioquia.
5. **Minifundio con mucho crédito** (142).
   - Es el segmento con más crédito por km²: 30 M$, el doble del siguiente.
   - Los hatos son muy pequeños (13 bovinos por finca), aunque hay bastante ganado por km² (51).
   - Tiene poco bosque (13 %) y la mayor área cosechada (11 %).
   - Son municipios pequeños (150 km²) de Boyacá, Cundinamarca y Santander.
6. **Palmero** (42).
   - La palma es el 67 % del área cosechada.
   - Tiene los hatos más grandes (116 bovinos por finca) y poco bosque (10 %).
   - Está en el Meta y en el Caribe: Cesar, Bolívar y Magdalena.
7. **Ganadero** (101).
   - Tiene la mayor densidad de ganado (69 bovinos por km²) y hatos grandes (73 por finca).
   - El 71 % del crédito va a la ganadería.
   - Casi no tiene bosque (8 %) y tiene poca agricultura (3 %).
   - Está sobre todo en el Caribe (Magdalena, Bolívar), en Boyacá y en Antioquia.
8. **Arrocero** (35).
   - El arroz es el 70 % del área cosechada.
   - Es el segmento con menos bosque (7 %).
   - Está en el Tolima, el Huila, Casanare y el Caribe.

Los perfiles tienen geografía, pero no son regiones. El ganadero y el de bosque con ganadería dispersa, por ejemplo,
están repartidos en cinco o seis regiones.

**Los componentes principales** resumen la mitad de la información (52 %) en dos ejes:

- El **componente 1** (32 %) va del bosque, a la izquierda, a la actividad agropecuaria intensa, a la derecha: más
  ganado por km², más crédito y más área cosechada. Es el eje que va de la selva al campo trabajado.
- El **componente 2** (20 %) separa dos cosas:
  - arriba, la ganadería de hatos grandes y los cultivos de llanura, como la palma y el arroz;
  - abajo, el café, el minifundio y el crédito por km².
- **Dónde queda cada segmento:**
  - la selva, sola a la izquierda;
  - el bosque con ganadería dispersa, entre la selva y el centro;
  - cafeteros y minifundio, abajo;
  - palmeros, arroceros y ganaderos, arriba a la derecha;
  - los cacaoteros, en el centro, mezclados con otros. En estos dos ejes no se distinguen porque el cacao pesa poco
    en los dos primeros componentes.

## 6. ¿Cuánto deforesta cada segmento?

**La pregunta.** ¿Los municipios con un mismo perfil productivo deforestan parecido? ¿Qué perfiles concentran la
deforestación?

**Cómo se hace.** La deforestación no se usó para formar los segmentos, así que esto es una **validación externa**:
si los segmentos deforestan distinto, el perfil productivo dice algo sobre la deforestación. Se usan tres medidas,
con el promedio de 2021 a 2025:

- **Tasa**: el % del bosque que se pierde cada año, según el IDEAM. Dice qué tan rápido se pierde el bosque que
  queda.
- **Hectáreas deforestadas por 100 km²**, según el IDEAM. Dice cuánta deforestación hay en el territorio.
- **Alertas de GFW por 100 km²**, como contraste con una fuente independiente.

**Cómo leer los gráficos.**

- **Cajas.** Cada caja va del primer al tercer cuartil y la raya es la mediana. La escala es logarítmica, pero
  incluye el 0.
- **Barras.** La gris es el % de municipios del segmento y la de color, el % de la deforestación que aporta. Si la de
  color es más larga, el segmento deforesta más de lo que le tocaría por su número de municipios.

In [ ]:
total = deforestacion.def_total_ha.sum()
mayores = deforestacion.def_total_ha.groupby(nombre_segmento).nlargest(3)
resumen = pd.DataFrame({
    "municipios": nombre_segmento.value_counts(),
    "% de los municipios": 100 * nombre_segmento.value_counts(normalize=True),
    f"% de la deforestación {ANIO_INICIO}-{ANIO_FIN}": 100 * deforestacion.def_total_ha.groupby(nombre_segmento).sum() / total,
    "tasa mediana (% del bosque por año)": deforestacion.tasa.groupby(nombre_segmento).median(),
    "mediana de ha deforestadas por 100 km² al año": deforestacion.def_100km2.groupby(nombre_segmento).median(),
    "mediana de alertas GFW (ha por 100 km² al año)": deforestacion.gfw_100km2.groupby(nombre_segmento).median(),
    "% de la deforestación del segmento en sus 5 municipios con más": deforestacion.def_total_ha.groupby(
        nombre_segmento).agg(lambda s: 100 * s.nlargest(5).sum() / s.sum()),
    "los 3 que más deforestan": {e: ", ".join(base.nombre[mayores[e].index]) for e in ETIQUETAS},
}).loc[ETIQUETAS]
print(f"Deforestación {ANIO_INICIO}-{ANIO_FIN} en los {len(X)} municipios: {total:,.0f} ha")
resumen.round(2)

In [ ]:
MEDIDAS = {"tasa": "Tasa de deforestación\n(% del bosque por año)",
           "def_100km2": "Deforestación IDEAM\n(ha por 100 km² al año)",
           "gfw_100km2": "Alertas de GFW\n(ha por 100 km² al año)"}
largo = deforestacion[list(MEDIDAS)].assign(segmento=nombre_segmento)
fig, axes = plt.subplots(1, 3, figsize=(22, 9), sharey=True)
for ax, (c, titulo) in zip(axes, MEDIDAS.items()):
    sns.boxplot(data=largo, x=c, y="segmento", order=ETIQUETAS, palette=COLOR_NOMBRE, fliersize=3, ax=ax)
    ax.set_xscale("symlog", linthresh=0.01)
    ax.set_xlim(left=0)
    ax.set(title=titulo, xlabel="escala logarítmica (con el 0)", ylabel="")
fig.suptitle("La deforestación en cada segmento (no se usó para formar los segmentos)", weight="bold", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .95))
plt.show()

In [ ]:
participacion = resumen[["% de los municipios", f"% de la deforestación {ANIO_INICIO}-{ANIO_FIN}"]]
fig, ax = plt.subplots(figsize=(14, 8))
alto = 0.38
posiciones = np.arange(K)
ax.barh(posiciones - alto / 2, participacion["% de los municipios"], height=alto, color="#bdbdbd",
        label="% de los municipios")
ax.barh(posiciones + alto / 2, participacion[f"% de la deforestación {ANIO_INICIO}-{ANIO_FIN}"], height=alto, color="#4d4d4d",
        label=f"% de la deforestación {ANIO_INICIO}-{ANIO_FIN}")
for i, (m, d) in enumerate(participacion.values):
    ax.text(m + 0.5, i - alto / 2, f"{m:.0f} %", va="center", fontsize=11)
    ax.text(d + 0.5, i + alto / 2, f"{d:.1f} %".replace(".", ",") if d < 1 else f"{d:.0f} %", va="center",
            fontsize=11, weight="bold")
ax.set(yticks=posiciones, yticklabels=ETIQUETAS, xlabel="%",
       title="Cuántos municipios tiene cada segmento y cuánta deforestación aporta")
ax.invert_yaxis()
ax.legend(loc="lower right")
plt.show()

**Qué se ve.**

- **La deforestación se concentra en cuatro perfiles.**
  - El bosque con ganadería dispersa (53 %), el palmero (14 %), el ganadero (13 %) y la selva (12 %) suman el 92 %
    de la deforestación con el 46 % de los municipios.
  - En el otro extremo, el cafetero y el minifundio tienen el 41 % de los municipios y solo el 1,4 % de la
    deforestación.
- **El segmento de bosque con ganadería dispersa aporta más de la mitad con el 13 % de los municipios.**
  - Ahí están Cartagena del Chairá, La Macarena, Calamar, San José del Guaviare, Solano y Puerto Guzmán: el arco de
    deforestación de la Amazonía.
  - Sus 5 municipios con más deforestación suman el 56 % del segmento.
- **Dentro de cada segmento, pocos municipios pesan mucho.**
  - En el palmero, 5 municipios suman el 87 % de la deforestación del segmento, con Mapiripán y Tibú a la cabeza.
  - En el ganadero suman el 83 %, con San Vicente del Caguán a la cabeza.
  - Por eso la mediana del segmento suele ser mucho más baja de lo que sugiere su total: la mayoría de los ganaderos
    deforesta poco, y unos pocos ganaderos de frontera deforestan muchísimo.
  - Las cajas son anchas por lo mismo. El segmento dice algo sobre cuánto deforesta un municipio, pero no lo dice
    todo.
- **La tasa cuenta otra historia.**
  - Los que pierden más rápido el bosque que les queda son el palmero (0,28 % al año, mediana), el arrocero
    (0,26 %) y el cacaotero (0,23 %).
  - El arrocero aporta poco en hectáreas porque le queda poco bosque (7 %), pero lo pierde rápido.
  - La selva y el bosque con ganadería dispersa tienen una tasa baja (0,08 %): pierden muchas hectáreas, pero tienen
    muchísimo bosque.
- **El cacaotero es el que más deforesta por km².** Su mediana es de 8,7 ha por 100 km² al año, y también es el que
  más alertas de GFW tiene (14,3). Coincide con las correlaciones de `09_eda_panel_municipal`: el cacao era
  la única variable asociada con más deforestación en los cinco grupos de tamaño de municipio.
- **Las dos fuentes coinciden.** Las alertas de GFW ordenan los segmentos casi igual que el IDEAM:
  - arriba, el cacaotero, el bosque con ganadería dispersa y la selva;
  - abajo, el minifundio, el ganadero y el arrocero.

**Lectura: qué hacer con cada perfil (insight por grupo para ODM-2 y OBJ-N1).**

*Son propuestas de uso, derivadas de las cifras de este notebook; no son resultados medidos ni prueban causalidad.* Las cifras son medianas del segmento salvo que se diga otra cosa. Crédito = M$ por km² al año y, entre paréntesis, la parte que va a ganadería.

**Donde está el volumen de hectáreas (92 % del total en cuatro perfiles)**

- **Bosque con ganadería dispersa** (86 municipios, 53 % de las ha, tasa 0,08 %, crédito 4,8 con 40 % ganadero; frontera difusa). Es el perfil de **mayor prioridad de monitoreo** por volumen: sus 5 municipios más deforestadores concentran el 56 % del segmento
  (Cartagena del Chairá, La Macarena y Calamar a la cabeza). Llega poco crédito y casi la mitad es ganadero, así que es donde una intervención preventiva (ordenar el crédito ganadero, trazabilidad) tendría más margen. Como su frontera es difusa, **la etiqueta no basta por sí sola**: conviene mirar municipio a municipio.
- **Palmero** (42, 14 % de las ha, tasa 0,28 %, la más alta, crédito 7,9 con 37 % ganadero; muy firme). Pocos municipios mandan: 5 suman el 87 % (Mapiripán, Tibú y Puerto Rico). Se trabaja **municipio a municipio**; el segmento sirve para saber que la palma, el hato grande (116 bovinos por finca)
  y el poco bosque (10 %) vienen juntos.
- **Ganadero** (101, 13 % de las ha, tasa 0,14 %, crédito 8,4 con 71 % ganadero; frontera difusa con el de bosque). Es el perfil donde el **cruce crédito ganadero-deforestación es más directo** (ODM-2): casi todo el crédito es ganadero. Pero la mediana deforesta poco (0,79 ha por 100 km² al año) y 5 municipios suman el 83 %
  (San Vicente del Caguán, Remedios, Yondó): la mayoría de los ganaderos no deforesta, y unos pocos de frontera sí.
- **Selva sin actividad agropecuaria** (64, 12 % de las ha, tasa 0,08 %, crédito 0,3 con 0 % ganadero; muy firme). Deforesta sin ganadería, cultivos ni crédito visibles, así que **el financiamiento no la explica y no es el instrumento**: aquí corresponde monitoreo satelital y gobernanza territorial.
  Los ceros pueden ser vacío de información (18 de los 64 son corregimientos departamentales).

**Donde hay poco volumen pero señal**

- **Cacao y bosque** (47, 4,5 % de las ha, tasa 0,23 %, crédito 10,4 con 35 % ganadero; intermedio). Es el que más deforesta **por km²** (8,7 ha por 100 km² al año) y el de más alertas de GFW (14,3).
  El cacao es un producto regulado por la EUDR, así que es el perfil donde la **trazabilidad de la cadena** tiene más sentido comercial. Ojo con la coca, que la EVA no mide.
- **Arrocero** (35, 1,8 % de las ha, tasa 0,26 %, crédito 13,5 con 23 % ganadero; firme). Pierde rápido el poco bosque que le queda (7 %): se vigila la **tasa**, no el volumen.

**Perfiles de contraste (menos del 2 % de las hectáreas)**

- **Cafetero** (123, 1,0 % de las ha, tasa 0,03 %, crédito 14,7) y **Minifundio con mucho crédito** (142, 0,4 % de las ha, tasa 0,01 %, crédito 30,4). Son el 41 % de los municipios y casi no deforestan, con el crédito más alto. Sirven de **grupo de comparación**: muestran que más crédito no
  coincide con más deforestación, pero tampoco prueban que el crédito proteja, porque son agricultura consolidada de cordillera con poco bosque.

**Mensaje común.** Cuanto más deforestación hay, menos crédito de producción llega (0,3 a 8,4 M$ en los cuatro perfiles de mayor volumen), y el que llega es sobre todo ganadero. Para el Observatorio, esto reorienta la pregunta: no es "dónde hay más crédito", sino **en qué tipo de municipio el crédito está o no está presente**.

## 7. Mapa de los segmentos

**Cómo se hace.** Se usan los límites municipales del DANE (Marco Geoestadístico Nacional 2025). Va un mapa por
segmento, en lugar de uno solo con siete colores, porque siete colores juntos no se distinguen bien, y menos con
daltonismo.

**Cómo leer el mapa.**

- En color, los municipios del segmento.
- En gris medio, los municipios rurales de otros segmentos.
- En gris claro, los que no son rurales y no entraron en la segmentación.

In [ ]:
# Límites de los municipios para el mapa: DANE, Marco Geoestadístico
# Nacional 2025, versión "Gráfico" (CC BY 4.0). Se descargan aquí, en
# memoria, y no se guardan en BigQuery: el Observatorio ya los tiene.
import requests
_r = requests.get(
    "https://services.arcgis.com/ioNRSZMYYlx0PUaB/arcgis/rest/services/"
    "MGN2025_MPIO_GRAFICO/FeatureServer/0/query",
    params={"where": "1=1", "outFields": "mpio_cdpmp,mpio_cnmbr", "outSR": "4326", "f": "geojson"},
    timeout=300)
_r.raise_for_status()
limites = gpd.GeoDataFrame.from_features(_r.json()["features"], crs="EPSG:4326")
# Simplificados a unos 50 m: alcanza para el mapa y se dibuja más rápido.
limites["geometry"] = limites.geometry.simplify(0.0005, preserve_topology=True)
limites = limites[["mpio_cdpmp", "geometry"]]
continental = limites[~limites.mpio_cdpmp.str.startswith("88")]          # sin San Andrés: no tiene rurales
mapa = continental.merge(segmento, left_on="mpio_cdpmp", right_index=True, how="left")
print(f"Municipios de la segmentación sin polígono en el mapa: {sorted(set(segmento.index) - set(limites.mpio_cdpmp))}")

fig, axes = plt.subplots(2, 4, figsize=(24, 18))
for ax, s in zip(axes.flat, SEGS):
    mapa[mapa.segmento.isna()].plot(ax=ax, color="#f4f4f4", edgecolor="#d0d0d0", linewidth=0.1)
    mapa[mapa.segmento.notna() & (mapa.segmento != s)].plot(ax=ax, color="#d4d4d4", edgecolor="#bdbdbd",
                                                            linewidth=0.1)
    mapa[mapa.segmento == s].plot(ax=ax, color=COLOR[s], edgecolor="white", linewidth=0.2)
    ax.set_title(f"{SEGMENTOS[s]}\n({(segmento == s).sum()} municipios)", fontsize=14)
    ax.set_axis_off()
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ("#d4d4d4", "#f4f4f4")],
           labels=["municipio rural de otro segmento", "municipio no rural (fuera de la segmentación)"],
           loc="lower center", ncol=2, fontsize=14, frameon=False)
fig.suptitle("Dónde está cada segmento", weight="bold", fontsize=17)
plt.tight_layout(rect=(0, 0.03, 1, .97))
plt.show()

**Qué se ve.**

- La **selva** está en el Chocó, en la costa nariñense y en la Amazonía y la Orinoquía orientales: Amazonas, Guainía,
  Vaupés y Vichada.
- El **bosque con ganadería dispersa** forma un arco en el piedemonte amazónico (Caquetá, Guaviare, sur del Meta y
  Putumayo), justo entre la selva y las zonas ganaderas. Además aparece en manchas a lo largo de los Andes y de la
  costa Caribe.
- El **cafetero** y el **minifundio** ocupan la cordillera:
  - el cafetero, desde Nariño y Cauca hasta Antioquia y Santander;
  - el minifundio, en el altiplano cundiboyacense, en Santander y en el sur de Nariño.
- El **palmero**, el **ganadero** y el **arrocero** están en las tierras bajas:
  - el palmero, en el Meta, el Magdalena medio y el Caribe;
  - el ganadero, en el Caribe, el Magdalena medio, Arauca y San Vicente del Caguán;
  - el arrocero, en el Tolima, el Huila, Casanare y el bajo Cauca.

El mapa de la deforestación de `09_eda_panel_municipal` muestra el mismo arco amazónico. Es la zona donde el
segmento de bosque con ganadería dispersa toca a la selva.

## 8. Conclusiones y limitaciones

### Qué se puede decir

1. **Los municipios rurales sí forman grupos, pero sus fronteras no son nítidas.**
   - La silueta (0,26) duplica la de los datos sin estructura, pero indica una estructura débil.
   - Los 8 segmentos son estables frente al remuestreo, con un Jaccard de 0,81 a 0,97.
   - Los más nítidos son la selva y el palmero. Los menos nítidos, el ganadero y el de bosque con ganadería dispersa.
2. **La deforestación no está donde está el crédito.**
   - Los cuatro segmentos que más deforestan son los cuatro que menos crédito de producción reciben por km²: de 0,3 a
     8,4 M$.
   - Los que más crédito reciben, el minifundio (30 M$) y el cafetero (15 M$), casi no deforestan.
   - Donde hay deforestación, el poco crédito que llega va sobre todo a la ganadería: el 40 % en el segmento de
     bosque con ganadería dispersa y el 71 % en el ganadero.
   - Esto no quiere decir que el crédito proteja el bosque. Los segmentos con más crédito son de agricultura
     consolidada en la cordillera. Los que más deforestan tienen mucho bosque (la selva y el de bosque con ganadería
     dispersa) o tienen ganadería extensiva y palma. La diferencia puede venir del tipo de actividad, y no del
     crédito en sí.
   - Tampoco es solo cuestión de cuánto bosque queda: el palmero (10 % de bosque) y el ganadero (8 %) tienen menos
     bosque que el minifundio (13 %), y aun así deforestan mucho más.
3. **La deforestación se concentra en pocos perfiles.**
   - El bosque con ganadería dispersa (53 %), el palmero (14 %), el ganadero (13 %) y la selva (12 %) suman el 92 %
     con menos de la mitad de los municipios.
   - El cafetero y el minifundio, con el 41 % de los municipios, casi no deforestan (1,4 %).
4. **Las hectáreas y la tasa dicen cosas distintas.**
   - Las hectáreas se concentran donde hay mucho bosque y algo de ganado, sobre todo en el arco amazónico.
   - La tasa más alta está en los segmentos de palma, arroz y cacao, que pierden más rápido el bosque que les queda.
     Al arrocero le queda poco bosque (7 %), así que aporta pocas hectáreas (1,8 %) aunque lo pierda rápido.
5. **Dentro de cada segmento, unos pocos municipios explican casi todo.** En el palmero y el ganadero, 5 municipios
   suman más del 80 % de la deforestación del segmento. El segmento no reemplaza el análisis municipio a municipio.
6. **El cacao aparece otra vez.** El segmento cacaotero tiene la mayor deforestación por km² y las mayores alertas
   de GFW por km² (medianas), en línea con las correlaciones del análisis del panel municipal.

### Limitaciones

- **Es descriptivo.** Que un segmento deforeste mucho no prueba que su perfil productivo cause la deforestación. El
  ganado y la palma, por ejemplo, pueden llegar después de la tala, no antes.
- **El resultado depende de decisiones de método.** Las participaciones de los cultivos van sin transformar, así que
  los segmentos se arman por el cultivo que domina en cada municipio. Otras transformaciones, que les den más peso a
  los cultivos pequeños, pueden armar segmentos distintos.
- **Todas las variables pesan lo mismo, y cuatro de las diez son cultivos.** Con otra selección de variables los
  segmentos pueden cambiar.
- **Faltan actividades importantes.** La EVA no mide pastos ni coca. La coca puede estar detrás de parte de la
  deforestación en segmentos como el cacaotero o el palmero, por ejemplo en Tibú o en el Valle del Guamuez.
- **Los ceros pueden ser falta de reporte.** En los corregimientos departamentales de la Amazonía, que no haya
  cultivos, ganado ni crédito puede ser un vacío de información y no una ausencia real.
- **El crédito está en pesos corrientes.** Como todos los municipios se miden en los mismos años, la comparación
  entre ellos no se afecta.

**Lectura: qué responde este notebook frente a los objetivos del documento, y qué no.**

- **ODM-2 (cruce de financiamiento y producción).** A nivel de tipo de municipio, la deforestación se concentra donde llega menos crédito de producción y donde el crédito, cuando llega, es ganadero (40 % en el perfil de bosque con ganadería dispersa y 71 % en el ganadero). El cruce fino por municipio queda fuera del alcance de este notebook.
- **ODM-3 (relación entre variables financieras, productivas y deforestación).** Es una **asociación descriptiva por perfil**. Un segmento no controla por el tamaño del bosque ni por la ubicación, así que no establece qué variable pesa más; dice qué combinaciones de actividad y crédito van con más o menos deforestación.
- **OBJ-N1 (capacidad de resolver problemas con esta data).** El aporte es un **mapa de tipos de municipio y de dónde actuar** (monitoreo, trazabilidad, crédito ganadero), con su nivel de solidez declarado.
- **REQ-04 y CDM-6.** Interpretable por construcción (diez variables, perfiles con nombre y cifras) y reproducible (semilla fija, `Pipeline`, estabilidad medida).
- **Lo que no demuestra.** (1) No es causal: la palma y el ganado pueden llegar después de la tala. (2) No es predictivo: no dice qué municipio deforestará el próximo año. (3) La estructura es **débil** (silueta 0,26) y el perfil más importante por hectáreas es el menos nítido; hay que reportarlo así. (4) Depende de las variables elegidas y de que los ceros sean reales.
- **Para el documento.** Reportar los 8 perfiles **con su solidez** (estabilidad y silueta) y con la estrategia propuesta por perfil (lectura anterior).

## Lo que pasa al notebook 17

En BigQuery solo quedan las tablas de datos, así que los resultados de la segmentación se ven aquí. Lo único
que se guarda es `SEGMENTACION_MUNICIPIOS`, una tabla de paso con un municipio rural por fila, su segmento, las
diez variables y su deforestación: la lee el notebook 17, que la borra al terminar. Si no se corre el 17,
vence sola a los 7 días.

La tabla de abajo es la de los clusters: un municipio por fila, con su cluster, el nombre y la descripción del
cluster y su deforestación, de más a menos deforestación dentro de cada cluster.

In [ ]:
salida = (base[["nombre", "departamento", "region", "ruralidad"]]
          .join(segmento).assign(nombre_segmento=nombre_segmento)
          .join(X.rename(columns=NOMBRES))
          .join(deforestacion.rename(columns={"tasa": "tasa de deforestación (% del bosque por año)",
                                              "def_100km2": "deforestación IDEAM (ha por 100 km² al año)",
                                              "gfw_100km2": "alertas GFW (ha por 100 km² al año)",
                                              "def_total_ha": f"deforestación IDEAM {ANIO_INICIO}-{ANIO_FIN} (ha)"}))
          .sort_values(["segmento", "nombre"]))
salida.pipe(guardar_resultado, "segmentacion_municipios", indice=True,
             descripcion="Segmentación: un municipio rural por fila, con su segmento, las diez variables de la "
             "segmentación y su deforestación (notebook 12).")


In [ ]:
# Tabla de clusters para la carpeta de resultados: un municipio por fila
DESCRIPCIONES = {
    1: "Casi todo el municipio es bosque. Casi no tiene ganado, cultivos ni crédito. Está en el Pacífico y en la "
       "Amazonía y la Orinoquía orientales.",
    2: "Mucho bosque con algo de ganado. Muy poca agricultura y poco crédito, buena parte para la ganadería. Incluye "
       "el piedemonte amazónico, donde se concentra la deforestación.",
    3: "El cacao es el cultivo principal y el municipio conserva bastante bosque.",
    4: "El café es más de la mitad del área cosechada. Está en la cordillera.",
    5: "Municipios pequeños de hatos muy pequeños, con la mayor área cosechada y el crédito por km² más alto. Poco "
       "bosque.",
    6: "La palma es el cultivo principal. Hatos muy grandes y poco bosque.",
    7: "La mayor densidad de ganado, en hatos grandes, y la mayor parte del crédito va a la ganadería. Casi no tiene "
       "bosque.",
    8: "El arroz es el cultivo principal. Es el segmento con menos bosque.",
}
clusters = pd.DataFrame({
    "municipio": base.nombre,
    "departamento": base.departamento,
    "cluster": segmento,
    "nombre del cluster": nombre_segmento.str[3:],
    "descripción del cluster": segmento.map(DESCRIPCIONES),
    f"deforestación IDEAM {ANIO_INICIO}-{ANIO_FIN} (ha)": deforestacion.def_total_ha,
    "deforestación IDEAM (ha por 100 km² al año)": deforestacion.def_100km2,
    "tasa de deforestación (% del bosque por año)": deforestacion.tasa,
}).rename_axis("cod_dane").sort_values(["cluster", f"deforestación IDEAM {ANIO_INICIO}-{ANIO_FIN} (ha)"], ascending=[True, False])

clusters.head(10)